In [1]:
!pip install -q langchain-groq langchain-core requests

In [ ]:
import os

groq_api_key = os.environ.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [3]:
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b, this tool returns their product."""
    return a * b


print(multiply.invoke({
    "a": 3,
    "b": 4
}))

print(multiply.name)
print(multiply.description)
print(multiply.args)

12
multiply
Given 2 numbers a and b, this tool returns their product.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.environ.get("GROQ_API_KEY")
)

llm.invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.052969908, 'completion_tokens_details': None, 'prompt_time': 0.004672662, 'prompt_tokens_details': None, 'queue_time': 0.162034322, 'total_time': 0.05764257}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0014a-210f-7541-bf4a-fffb6b207379-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

In [5]:
llm_with_tools = llm.bind_tools([multiply])

llm_with_tools.invoke("Hi, how are you?")

AIMessage(content="I'm doing well, thanks for asking. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 238, 'total_tokens': 264, 'completion_time': 0.0790755, 'completion_tokens_details': None, 'prompt_time': 0.012889868, 'prompt_tokens_details': None, 'queue_time': 0.161658675, 'total_time': 0.091965368}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0014a-31ec-7190-b63c-a4f3ebb40e0c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 238, 'output_tokens': 26, 'total_tokens': 264})

In [6]:
query = HumanMessage(
    "Can you multiply 3 with 1000?"
)

messages = [query]

result = llm_with_tools.invoke(messages)

result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 1000},
  'id': 'zssrc7jqm',
  'type': 'tool_call'}]

In [7]:
tool_result = multiply.invoke(result.tool_calls[0])

tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='zssrc7jqm')

In [8]:
messages.append(result)
messages.append(tool_result)

messages

[HumanMessage(content='Can you multiply 3 with 1000?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'zssrc7jqm', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 242, 'total_tokens': 262, 'completion_time': 0.063080828, 'completion_tokens_details': None, 'prompt_time': 0.023412534, 'prompt_tokens_details': None, 'queue_time': 0.162137478, 'total_time': 0.086493362}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0014a-65cc-7483-b462-30ade95f5c4f-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'zssrc7jqm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 242, 'output_tokens': 20, 'total_tokens': 262}),
 To

In [9]:
final_response = llm_with_tools.invoke(messages)

print(final_response.content)

The result of multiplying 3 by 1000 is 3000.


# Currency exchange

In [56]:
from dotenv import load_dotenv
import os

load_dotenv()

exchange_rate_api_key = os.environ.get("EXCHANGE_RATE_API_KEY")
groq_api_key = os.environ.get("GROQ_API_KEY")

In [24]:
import os
import requests

from typing import Annotated

from langchain_groq import ChatGroq

from langchain_core.tools import tool, InjectedToolArg

from langchain_core.messages import (
    HumanMessage,
    ToolMessage
)

In [25]:
print("Groq key exists:", bool(os.environ.get("GROQ_API_KEY")))
print("ExchangeRate key exists:", bool(os.environ.get("EXCHANGE_RATE_API_KEY")))

Groq key exists: True
ExchangeRate key exists: True


In [26]:
@tool
def get_conversion_factor(
    base_currency: str,
    target_currency: str
) -> dict:
    """
    Get the current exchange rate between
    a base currency and a target currency.
    """

    api_key = os.environ["EXCHANGE_RATE_API_KEY"]

    url = (
        f"https://v6.exchangerate-api.com/v6/"
        f"{api_key}/pair/"
        f"{base_currency}/{target_currency}"
    )

    response = requests.get(url)

    response.raise_for_status()

    return response.json()

In [27]:
usd_to_inr = get_conversion_factor.invoke({
    "base_currency": "USD",
    "target_currency": "INR"
})

print(usd_to_inr)

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1786665602, 'time_last_update_utc': 'Fri, 14 Aug 2026 00:00:02 +0000', 'time_next_update_unix': 1786752002, 'time_next_update_utc': 'Sat, 15 Aug 2026 00:00:02 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 95.4733}


In [29]:
@tool
def convert(
    base_currency_value: float,
    conversion_rate: Annotated[float, InjectedToolArg]
) -> float:
    """
    Convert an amount using the exchange rate
    obtained from the exchange rate API.
    """

    return base_currency_value * conversion_rate

In [30]:
print(convert.args)

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'number'}}


In [31]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [60]:
response = llm.invoke("Say hello in one sentence.")

print(response.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [33]:
llm_with_tools = llm.bind_tools(
    [
        get_conversion_factor,
        convert
    ],
    parallel_tool_calls=False
)

In [34]:
messages = [
    HumanMessage(
        "Convert 10 USD to INR"
    )
]

In [35]:
ai_message = llm_with_tools.invoke(messages)

print(ai_message.tool_calls)

[{'name': 'convert', 'args': {'base_currency': 'USD', 'base_currency_value': 10, 'target_currency': 'INR'}, 'id': '17wj16392', 'type': 'tool_call'}]


In [39]:
messages.append(ai_message)

In [47]:
tool_call = ai_message.tool_calls[0]

tool_result = get_conversion_factor.invoke(tool_call)

print(tool_result)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1786665602, "time_last_update_utc": "Fri, 14 Aug 2026 00:00:02 +0000", "time_next_update_unix": 1786752002, "time_next_update_utc": "Sat, 15 Aug 2026 00:00:02 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 95.4733}' name='get_conversion_factor' tool_call_id='17wj16392'


In [59]:
import json

data = json.loads(tool_result.content)

conversion_rate = data["conversion_rate"]

print("Actual API rate:", conversion_rate)

Actual API rate: 95.4733


In [58]:
amount = 99

converted_value = convert.invoke({
    "base_currency_value": amount,
    "conversion_rate": conversion_rate
})

print("Converted value:", converted_value)

Converted value: 9451.8567


In [57]:
print(
    f"The current conversion rate from USD to INR is "
    f"{conversion_rate}.\n\n"
    f"{amount} USD is equivalent to "
    f"{converted_value:.2f} INR."
)

The current conversion rate from USD to INR is 95.4733.

99 USD is equivalent to 9451.86 INR.
